# 📓 Semana 3 · Dia 6 — Pipeline de limpeza + simulado parcial DEA (Spark)

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (Spark) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Pipeline de limpeza rodando + simulado ≥ 70% |

---


## 📖 Teoria — O entregável: pipeline de limpeza (Bronze → pré-Prata)

Vamos consolidar a semana em um pipeline idempotente de limpeza — ele será a base da camada Prata na Semana 4.

**Idempotente** = rodar 2x dá o mesmo resultado. Regra de ouro da Prata.


### 💻 Na prática — Pipeline completo

Rode de ponta a ponta: leitura → limpeza → enriquecimento → escrita.


In [ ]:
# 1) Ler Bronze
from pyspark.sql.functions import col, when, to_timestamp, upper, round as r, trim
df = spark.table("workspace.bronze.vendas_bronze")
print("Bronze:", df.count())

In [ ]:
# 2) Limpeza: nulos, duplicatas, tipos, regras de negócio
df_limpo = (df
    .filter(col("CustomerID").isNotNull())
    .withColumn("Description", trim(col("Description")))
    .withColumn("Country", upper(trim(col("Country"))))
    .dropDuplicates(["InvoiceNo", "StockCode", "InvoiceDate"])
    .filter("Quantity > 0 AND UnitPrice > 0")
    .withColumn("receita_linha", r(col("Quantity") * col("UnitPrice"), 2))
    .withColumn("faixa",
        when(col("receita_linha") < 20, "baixa")
        .when(col("receita_linha") < 100, "media")
        .otherwise("alta")))
print("Após limpeza:", df_limpo.count())

In [ ]:
# 3) Auditoria rápida (as 6 dimensões)
print("Nulos restantes por coluna:")
df_limpo.select([count(isnull(c)).alias(c) for c in ["CustomerID", "Description"]]).show()
print("Receita total:", df_limpo.agg(s("receita_linha")).collect()[0][0])

In [ ]:
# 4) Gravar como tabela pré-Prata (idempotente)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.prata")
df_limpo.write.mode("overwrite").saveAsTable("workspace.prata.vendas_preprata")
print("Pré-Prata gravada:", spark.table("workspace.prata.vendas_preprata").count())

### 💻 Na prática — Simulado parcial — domínio Spark (10 questões)

Responda antes de ver o gabarito.


### Questões Spark

**1.** Qual é a diferença entre transformação e ação?
- A) transformação roda imediato; ação é lazy
- B) transformação é lazy; ação dispara execução
- C) iguais
- D) ação só funciona em RDD

**2.** Onde as tasks rodam?
- A) no driver  B) nos executores, sobre partições  C) no metastore  D) no SQL Warehouse

**3.** Qual operador é o mais caro?
- A) select  B) filter  C) shuffle  D) withColumn

**4.** Quando o broadcast join é aplicado automaticamente?
- A) sempre  B) quando a tabela é pequena (< threshold)  C) quando há índices  D) nunca

**5.** `df.cache()` materializa quando?
- A) na chamada  B) na primeira ação  C) no collect  D) nunca

**6.** Qual é a forma correta de criar coluna condicional?
- A) `df['col'] = ...`  B) `withColumn` + `when`  C) UDF obrigatório  D) `addColumn`

**7.** Qual comando lê um JSON com schema?
- A) `spark.read.schema(s).json(path)`  B) `spark.read.json(s)`  C) `spark.json(s)`  D) `LOAD`

**8.** O que `left anti` retorna?
- A) linhas da esquerda com match  B) linhas da esquerda SEM match  C) todas  D) nulas

**9.** Qual API é recomendada para lógica pandas distribuída?
- A) RDD  B) pandas API on Spark  C) UDF Python  D) SQL puro

**10.** O que o Spark UI mostra?
- A) DAG, estágios, tasks e tempos  B) apenas o schema  C) apenas custos  D) nada em serverless


## 📖 Teoria — Gabarito — Simulado Spark

**1-B** · **2-B** · **3-C** · **4-B** · **5-B** · **6-B** · **7-A** · **8-B** · **9-B** · **10-A**.

≥ 7 acertos = pronto para a Semana 4. Se errou shuffle/broadcast, revise o Dia 5.


> 🎯 **Dica de prova**: Lazy evaluation + ações + shuffle + broadcast é o 'quarteto' de performance que aparece em toda prova Spark/DEA. Decore o quarteto.


## 🎯 Exercícios de fixação

**1.** Explique por que o pipeline acima é idempotente.

**2.** O que aconteceria se rodássemos sem `mode('overwrite')` a segunda vez?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Idempotente

Porque recria a tabela do zero (overwrite) a partir da mesma fonte — o resultado final é sempre o mesmo, independente de quantas vezes rodar.

**2.** Sem overwrite

Falha por 'table already exists' (ou acumularia duplicatas com append). Idempotência exige overwrite ou merge.



## ✅ Checklist de fechamento

- [ ] Rodei todos os 6 notebooks da Semana 3.
- [ ] Expliquei lazy evaluation e DAG em 2 frases.
- [ ] Sei quando usar broadcast join e o que é shuffle.
- [ ] Fiz o simulado Spark e revisei os erros.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*